In [1]:

import os 
import sys

import pandas as pd
import numpy as np

from preproces_prod3 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data,analyze_vrs_data,match_nn_max_dist_weigths,comparar_medias_test, charly_mip, charly_double_mip, mylogit, results_match, tabla_marcel, tabla_final,summary_eff,tabla_1_match

import inv

from IPython.core.display import display
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

np.random.seed(42)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'


In [ ]:
df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_02_06_25.csv')

In [7]:
fecha_referencia_nacido_str = '2024-09-30'
df_vrs_match_case = df_vrs_match_case.assign(edad_relativa =  lambda x: (pd.to_datetime(fecha_referencia_nacido_str) - pd.to_datetime(x['fecha_nac'])).dt.days).copy()

In [10]:
path_data

WindowsPath('c:/Users/ntrig/Desktop/ISCI/Proyectos/Efectividad_Nirse/Data')

In [11]:
df_vrs_match_case.to_csv(path_data/'df_vrs_match_case_02_06_25_v2.csv')

In [20]:
df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_02_06_25_v2.csv')

In [23]:
with open(path_data/'lista_ruts_cardio.pkl', 'rb') as f:
    lista_ruts_cardio = pickle.load(f)

In [24]:
df = (df_vrs_match_case
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .assign(cardio1 = lambda x: x.RUN.isin(lista_ruts_cardio).astype(int),
            #   displa = lambda x: x.RUN.isin(ruts_displasia.RUN).astype(int),
              fecha_nac = lambda x: pd.to_datetime(x.fecha_nac, infer_datetime_format=True),
              year_nac = lambda x: x.fecha_nac.dt.year,
              month_nac = lambda x: x.fecha_nac.dt.month,
              criterio_pali_normal = lambda x: ((x.RUN.isin(lista_ruts_cardio)) | (x.SEMANAS<32) | (x.PESO<1500)).astype(int),
              pali = lambda x: np.where(x.criterio_pali_normal==1, 1, 
                                        np.where((x.year_nac==2023) & (x.month_nac.between(7, 9, inclusive='both')) & (x.SEMANAS<=34) & ((x.PESO<2500)), 1, 0)))
      .copy()
      )

In [25]:
df.loc[df.RUN == 'ac483764636448d753930868dd3192a785e3728a2d81b3fc44dba45c3506255a', 'region'] = 'METROPOLITANA'
df.loc[df.COMUNA == 12202, 'region'] = 'MAGALLANES Y ANTARTICA'

In [ ]:
df.query('(pali==1) | (SEMANAS<=35)').query('diag_vrs==1').DIAG1.value_counts()


DIAG1
J219    80
J210    48
J121    30
J205    21
Name: count, dtype: int64

In [6]:
def _tabla_1_single(
    df,
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    import numpy as np
    import pandas as pd

    df = df.copy()
    df['grupo_tabla'] = np.where(df[id_col] == df[group_col], case_label, control_label)
    df['sexo'] = df['sexo'].map({0: 'Female', 1: 'Male'})

    variables_numericas_mediana = ['edad_relativa']
    variables_numericas_media = ['SEMANAS', 'PESO']
    variables_categoricas = ['sexo']
    #macrozona_var = 'Macrozones'

    etiquetas_variables = {
        'edad_relativa': 'Age (days)',
        'SEMANAS': 'Gestational age at birth (weeks)',
        'PESO': 'Birth weight (grams)',
        'sexo': 'Sex',
    #    'Macrozones': 'Macro-zone',
        'cardio1': 'Congenital heart disease',
        'prematuro': 'Prematurity',
        'event_upc': 'PICU admission',
        'super_preterm':'Prematurity Extreme'
    }

    grupos = df['grupo_tabla'].unique()
    tabla = []
    index = []
    n_por_grupo = {g: len(df[df['grupo_tabla'] == g]) for g in grupos}
    nombres_columnas = {g: f"{g} (n = {n_por_grupo[g]})" for g in grupos}

    # ---- Variables numéricas con mediana (IQR)
    for var in variables_numericas_mediana:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            mediana = subset.median()
            iqr = subset.quantile(0.75) - subset.quantile(0.25)
            fila[nombres_columnas[g]] = f'{mediana:.{decimales}f} ({iqr:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

        # Rango por edad relativa: <90, 90-180, >180
        bins = [0, 90, 180, float('inf')]
        labels = ['<90', '90-180', '>180']
        df['edad_intervalo'] = pd.cut(df['edad_relativa'], bins=bins, labels=labels, right=False)
        for val in labels:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset['edad_intervalo'] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    # ---- Variables categóricas comunes
    for var in variables_categoricas:
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append({})  # Fila vacía para separar
        niveles = df[var].dropna().unique()
        for val in niveles:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset[var] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    # ---- Variables numéricas con media (SD)
    for var in variables_numericas_media:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            media = subset.mean()
            sd = subset.std()
            fila[nombres_columnas[g]] = f'{media:.{decimales}f} ({sd:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

        if var == 'SEMANAS':
            # Rango para SEMANAS: 24-28 y 29-35
            bins = [0, 29, 36]  # hasta <29 y <36
            labels = ['24-28', '29-35']
            df['semana_intervalo'] = pd.cut(df['SEMANAS'], bins=bins, labels=labels, right=False)
            for val in labels:
                fila = {}
                for g in grupos:
                    subset = df[df['grupo_tabla'] == g]
                    total = len(subset)
                    cuenta = (subset['semana_intervalo'] == val).sum()
                    porcentaje = 100 * cuenta / total if total > 0 else 0
                    fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
                index.append((etiquetas_variables.get(var, var), val))
                tabla.append(fila)
    index.append(('Risk groups', ''))
    tabla.append({})
    for var in ['cardio1', 'prematuro','super_preterm']:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[var] == 1).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append(('Risk groups', etiquetas_variables.get(var, var)))
        tabla.append(fila)

    # ---- Macrozonas
    # index.append((etiquetas_variables.get(macrozona_var, macrozona_var), ''))
    # tabla.append({})
    # niveles = df[macrozona_var].dropna().unique()
    # for val in niveles:
    #     fila = {}
    #     for g in grupos:
    #         subset = df[df['grupo_tabla'] == g]
    #         total = len(subset)
    #         cuenta = (subset[macrozona_var] == val).sum()
    #         porcentaje = 100 * cuenta / total if total > 0 else 0
    #         fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    #     index.append((etiquetas_variables.get(macrozona_var, macrozona_var), val))
    #     tabla.append(fila)

    # ---- PICU
    fila = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g]
        total = len(subset)
        cuenta = (subset['cama'] == 'UPC').sum()
        porcentaje = 100 * cuenta / total if total > 0 else 0
        fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    index.append((etiquetas_variables.get('cama', 'PICU'), ''))
    tabla.append(fila)

    # -------------------------------------------------------------------------
    # A) Nirsevimab Recipients (inmunizado=1 => "Yes", inmunizado=0 => "No")
    # -------------------------------------------------------------------------
    # Agregamos un bloque para la variable 'inmunizado'
    index.append(("Nirsevimab Recipients", ""))
    tabla.append({})  # Fila vacía para separar visualmente

    # 1. Bloque categórico "Yes / No"
    for val_inm, label_inm in [(1, "Yes"), (0, "No")]:
        fila_inm = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            if 'inmunizado' not in subset.columns:
                # Si la columna no existe, forzamos 0
                fila_inm[nombres_columnas[g]] = "- (No data)"
            else:
                cuenta = (subset['inmunizado'] == val_inm).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila_inm[nombres_columnas[g]] = f"{cuenta} ({porcentaje:.1f}%)"
        index.append(("Nirsevimab Recipients", label_inm))
        tabla.append(fila_inm)

    # 2. Bloque "Time immunized (days)"
    #    Resta 'fechaIng_any' - 'fecha_Inm'. Si no hay datos, "-", sino media en días

    # -------------------------------------------------------------------------
    # ---- Diagnósticos - 2 grupos: diag_labels y diag_agrupao
    # -------------------------------------------------------------------------
    diag_labels = {
        'A090': 'Infectious diarrhoea',
        'R681': 'Other general symptoms and signs',
        'R11X': 'Nausea and vomiting',
        'R104': 'Abdominal pain',
        'S099': 'Other head injuries',
        'R634': 'Feeding difficulties',
        'R633': 'Polydipsia',
        'N390': 'Urinary tract infection',
        'P599': 'Other neonatal jaundice',
    }

    diag_agrupao = {
        'A099': 'Other gastroenteritis and colitis',
        'A080': 'Other gastroenteritis and colitis',
        'A083': 'Other gastroenteritis and colitis',
        'A082': 'Other gastroenteritis and colitis',
        'A084': 'Other gastroenteritis and colitis',
        'A085': 'Other gastroenteritis and colitis',
        'A081': 'Other gastroenteritis and colitis',
    }

    diag_frecuencias = []
    # for diag_code, label in diag_labels.items():
    #     fila_diag = {}
    #     count_control = 0
    #     for g in grupos:
    #         if g == case_label:
    #             fila_diag[nombres_columnas[g]] = '-'  # Casos no exhiben "motivo"
    #         else:
    #             subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
    #             total = len(df[df['grupo_tabla'] == g])
    #             cuenta = len(subset)
    #             count_control = cuenta
    #             porcentaje = 100 * cuenta / total if total > 0 else 0
    #             fila_diag[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    #     diag_frecuencias.append((count_control, ('Reason for hospital visit (controls)', label), fila_diag))

    # # Agrupación multiple: todos esos A09x a 'Other gastroenteritis and colitis'
    # # calculamos la suma total para los 'controls'
    # count_control_agrupao = 0
    # for diag_code, label in diag_agrupao.items():
    #     for g in grupos:
    #         if g == control_label:
    #             subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
    #             total = len(df[df['grupo_tabla'] == g])
    #             cuenta = len(subset)
    #             count_control_agrupao += cuenta

    # fila_agrupao = {}
    # porcentaje_agrupao = 100 * count_control_agrupao / total if total > 0 else 0
    # # Asignamos al control
    # fila_agrupao[nombres_columnas[control_label]] = f'{count_control_agrupao} ({porcentaje_agrupao:.1f}%)'
    # # Casos sin "motivo"
    # fila_agrupao[nombres_columnas[case_label]] = '-'
    # diag_frecuencias.append((
    #     count_control_agrupao,
    #     ('Reason for hospital visit (controls)', 'Other gastroenteritis and colitis'),
    #     fila_agrupao
    # ))

    # # Insertamos encabezado en la tabla
    # index.append(('Reason for hospital visit (controls)', ''))
    # tabla.append({})

    # # Ordenar por conteo y actualizar la tabla
    # for _, idx_, fila_ in sorted(diag_frecuencias, key=lambda x: x[0], reverse=True):
    #     index.append(idx_)
    #     tabla.append(fila_)

    multiindex = pd.MultiIndex.from_tuples(index, names=['Variable', 'Value'])
    return pd.DataFrame(tabla, index=multiindex)

def tabla_1_match_multi(
    df1=None, label1="base1",
    df2=None, label2="base2",
    df3=None, label3="base3",
    df4=None, label4="base4",
    df5=None, label5="base5",
    df6=None, label6="base6",
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    """
    Genera la tabla 1 para hasta 6 bases de datos. Si se pasa sólo df1, 
    reproduce la tabla de la función original. Para 2 a 6, concatena 
    horizontalmente las tablas.
    """
    # Lista de (df, label) dinámica
    df_list = []
    for df, lbl in [
        (df1, label1), (df2, label2), (df3, label3),
        (df4, label4), (df5, label5), (df6, label6)
    ]:
        if df is not None:
            df_list.append((df, lbl))

    if not df_list:
        print("No se recibió ninguna base de datos.")
        return None

    # Procesar cada base y renombrar columnas
    tables = []
    for df_x, lbl in df_list:
        single_table = _tabla_1_single(
            df_x,
            id_col=id_col,
            group_col=group_col,
            case_label=case_label,
            control_label=control_label,
            decimales=decimales
        )
        # MultiIndex para evitar colisiones
        single_table.columns = pd.MultiIndex.from_product([[lbl], single_table.columns])
        tables.append(single_table)

    # Devolver resultado según número de tablas
    if len(tables) == 1:
        return tables[0]
    else:
        return pd.concat(tables, axis=1)

In [7]:
df3 = (pd.read_csv(path_data/'matched_data_pali_all_born.csv',index_col=0)
       .assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days))
df2 = (pd.read_csv(path_data/'matched_data_prematuros_no_pali_all_born.csv',index_col=0)
       .assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days))
df1 = pd.concat([df2,df3])
df4 = (pd.read_csv(path_data/'matched_data_super_prematuros_all_born.csv',index_col=0)
       .assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days))
df5 = (pd.read_csv(path_data/'matched_data_cardiopats_1_all_born.csv',index_col=0)
       .assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days))


Tabla_1 = tabla_1_match_multi(
    df1=df1, label1="Pali + No-Pali",
    df2=df2, label2="Preterms-No-Pali",
    df3=df3, label3="Pali",
    df4=df4, label4="Super-Preterm",
    df5=df5, label5="Heart-congenital-disease",
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
)

In [9]:
Tabla_1.columns

MultiIndex([(          'Pali + No-Pali',     'Caso (n = 173)'),
            (          'Pali + No-Pali', 'Control (n = 1211)'),
            (        'Preterms-No-Pali',      'Caso (n = 87)'),
            (        'Preterms-No-Pali',  'Control (n = 609)'),
            (                    'Pali',      'Caso (n = 86)'),
            (                    'Pali',  'Control (n = 602)'),
            (           'Super-Preterm',      'Caso (n = 54)'),
            (           'Super-Preterm',  'Control (n = 378)'),
            ('Heart-congenital-disease',      'Caso (n = 35)'),
            ('Heart-congenital-disease',  'Control (n = 245)')],
           )

In [18]:
tabla_1_flat

,Unnamed: 0,Unnamed: 1,Pali + No-Pali,Pali + No-Pali.1,Preterms-No-Pali,Preterms-No-Pali.1,Pali,Pali.1,Super-Preterm,Super-Preterm.1,Heart-congenital-disease,Heart-congenital-disease.1
0,NaN,NaN,Caso (n = 177),Control (n = 708),Caso (n = 87),Control (n = 348),Caso (n = 90),Control (n = 360),Caso (n = 55),Control (n = 220),Caso (n = 39),Control (n = 156)
1,Variable,Value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Age (days),NaN,210.00 (118.00),210.50 (120.75),190.00 (123.00),191.00 (123.00),227.00 (106.75),230.50 (113.25),248.00 (116.00),249.00 (132.00),204.00 (68.50),201.50 (79.50)
3,Age (days),<90,13 (7.3%),56 (7.9%),12 (13.8%),47 (13.5%),1 (1.1%),9 (2.5%),0 (0.0%),2 (0.9%),1 (2.6%),8 (5.1%)
4,Age (days),90-180,49 (27.7%),203 (28.7%),26 (29.9%),105 (30.2%),23 (25.6%),98 (27.2%),13 (23.6%),54 (24.5%),11 (28.2%),47 (30.1%)
5,Age (days),>180,115 (65.0%),449 (63.4%),49 (56.3%),196 (56.3%),66 (73.3%),253 (70.3%),42 (76.4%),164 (74.5%),27 (69.2%),101 (64.7%)
6,Sex,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Sex,Male,109 (61.6%),393 (55.5%),55 (63.2%),196 (56.3%),54 (60.0%),197 (54.7%),32 (58.2%),125 (56.8%),23 (59.0%),80 (51.3%)
8,Sex,Female,68 (38.4%),315 (44.5%),32 (36.8%),152 (43.7%),36 (40.0%),163 (45.3%),23 (41.8%),95 (43.2%),16 (41.0%),76 (48.7%)
9,Gestational age at birth (weeks),NaN,33.03 (3.46),33.28 (3.26),33.95 (1.10),34.04 (1.01),32.14 (4.57),32.54 (4.34),29.00 (2.31),29.53 (2.22),36.13 (3.65),36.49 (3.14)


In [19]:
from scipy.stats import chi2_contingency, fisher_exact, ttest_ind_from_stats
import numpy as np
import pandas as pd
import re

def calcular_pvalores_csv(tabla_csv: pd.DataFrame, grupo: str) -> pd.DataFrame:
    col_caso = grupo
    col_control = grupo + ".1"

    var_chi2 = {
        "Age (days)": ["<90", "90-180", ">180"],
        "Sex": ["Male", "Female"],
        "Gestational age at birth (weeks)": ["24-28", "29-35"],
        "Risk groups": ["Congenital heart disease", "Prematurity", "Prematurity Extreme"],
        "Nirsevimab Recipients": ["Yes", "No"]
    }

    var_cont = "Birth weight (grams)"

    def extraer_n(valor):
        if pd.isna(valor):
            return 0
        try:
            return int(valor.split("(")[0].strip())
        except:
            return 0

    def extraer_media_sd(valor):
        try:
            media, sd = valor.strip(")").split(" (")
            return float(media), float(sd)
        except:
            return np.nan, np.nan

    def extraer_n_desde_header(valor):
        match = re.search(r"n\s*=\s*(\d+)", valor)
        return int(match.group(1)) if match else 0

    # Extraer n de cada grupo desde la primera fila (fila 0)
    n_caso = extraer_n_desde_header(tabla_csv[col_caso].iloc[0])
    n_control = extraer_n_desde_header(tabla_csv[col_control].iloc[0])

    p_values = []

    for variable, categorias in var_chi2.items():
        freqs = []
        for cat in categorias:
            fila = tabla_csv[(tabla_csv["Unnamed: 0"] == variable) & (tabla_csv["Unnamed: 1"] == cat)]
            if not fila.empty:
                nc = extraer_n(fila[col_caso].values[0])
                nt = extraer_n(fila[col_control].values[0])
                freqs.append([nc, nt])
            else:
                freqs.append([0, 0])

        matrix = np.array(freqs).T

        if matrix.shape == (2, 2):
            if np.any(matrix == 0):
                try:
                    _, pval = fisher_exact(matrix)
                except:
                    pval = np.nan
            else:
                try:
                    _, pval, _, _ = chi2_contingency(matrix)
                except:
                    pval = np.nan
        else:
            if np.any(matrix.sum(axis=0) == 0) or np.any(matrix.sum(axis=1) == 0):
                pval = np.nan
            else:
                try:
                    _, pval, _, _ = chi2_contingency(matrix)
                except:
                    pval = np.nan

        p_values.append((variable, pval))

    # Birth weight
    fila_bw = tabla_csv[(tabla_csv["Unnamed: 0"] == var_cont) & tabla_csv["Unnamed: 1"].isna()]
    if not fila_bw.empty:
        media1, sd1 = extraer_media_sd(fila_bw[col_caso].values[0])
        media2, sd2 = extraer_media_sd(fila_bw[col_control].values[0])
        try:
            _, pval_bw = ttest_ind_from_stats(mean1=media1, std1=sd1, nobs1=n_caso,
                                              mean2=media2, std2=sd2, nobs2=n_control,
                                              equal_var=False)
        except:
            pval_bw = np.nan
    else:
        pval_bw = np.nan

    p_values.append((var_cont, pval_bw))

    return pd.DataFrame(p_values, columns=["Variable", "p-value"])


tabla_1_flat = pd.read_csv(path_data/'tabla_1_match_multi_rev.csv')

df_pvals = calcular_pvalores_csv(tabla_1_flat, grupo="Pali + No-Pali")

with pd.ExcelWriter(path_data / "tabla_p_valores_actualizada.xlsx", engine="openpyxl") as writer:
    df_pvals.to_excel(writer, sheet_name="tabla_p_values", index=False)